# Data Engineer Certification - Practical Exam - Supplement Experiments

1001-Experiments makes personalized supplements tailored to individual health needs.

1001-Experiments aims to enhance personal health by using data from wearable devices and health apps.

This data, combined with user feedback and habits, is used to analyze and refine the effectiveness of the supplements provided to the user through multiple small experiments.

The data engineering team at 1001-Experiments plays a crucial role in ensuring the collected health and activity data from thousands of users is accurately organized and integrated with the data from supplement usage. 

This integration helps 1001-Experiments provide more targeted health and wellness recommendations and improve supplement formulations.


## Task

1001-Experiments currently has the following four datasets with four months of data:
 - "user_health_data.csv" which logs daily health metrics, habits and data from wearable devices,
 - "supplement_usage.csv" which records details on supplement intake per user,
 - "experiments.csv" which contains metadata on experiments, and
 - "user_profiles.csv" which contains demographic and contact information of the users.

Each dataset contains unique identifiers for users and/or their supplement regimen.

The developers and data scientsits currently manage code that cross-references all of these data sources separately, which is cumbersome and error-prone.

Your manager has asked you to write a Python function that cleans and merges these datasets into a single dataset.

The final dataset should provide a comprehensive view of each user's health metrics, supplement usage, and demographic information.

- To test your code, your manager will run only the code `merge_all_data('user_health_data.csv', 'supplement_usage.csv', 'experiments.csv', 'user_profiles.csv')`
- Your `merge_all_data` function must return a DataFrame, with columns as described below.
- All columns must accurately match the descriptions provided below, including names.


## Data

The provided data is structured as follows:

![database schema](schema.png)

The function you write should return data as described below.

There should be a unique row for each daily entry combining health metrics and supplement usage.

Where missing values are permitted, they should be in the default Python format unless stated otherwise.

| Column Name        | Description |
|--------------------|-------------|
| user_id            | Unique identifier for each user. </br>There should not be any missing values. |
| date               | The date the health data was recorded or the supplement was taken, in date format. </br>There should not be any missing values. |
| email              | Contact email of the user. </br>There should not be any missing values. |
| user_age_group  | The age group of the user, one of: 'Under 18', '18-25', '26-35', '36-45', '46-55', '56-65', 'Over 65' or 'Unknown' where the age is missing.|
| experiment_name    | Name of the experiment associated with the supplement usage. </br>Missing values for users that have user health data only is permitted. |
| supplement_name    | The name of the supplement taken on that day. Multiple entries are permitted. </br>Days without supplement intake should be encoded as 'No intake'. |
| dosage_grams       | The dosage of the supplement taken in grams. Where the dosage is recorded in mg it should be converted by division by 1000.</br>Missing values for days without supplement intake are permitted. |
| is_placebo         | Indicator if the supplement was a placebo (true/false). </br>Missing values for days without supplement intake are permitted. |
| average_heart_rate | Average heart rate as recorded by the wearable device. </br>Missing values are permitted. |
| average_glucose    | Average glucose levels as recorded on the wearable device. </br>Missing values are permitted. |
| sleep_hours        | Total sleep in hours for the night preceding the current day’s log. </br>Missing values are permitted. |
| activity_level     | Activity level score between 0-100. </br>Missing values are permitted. |

In [2]:
import pandas as pd
import numpy as np

def merge_all_data(health_path, supplement_path, experiments_path, profiles_path):
    # Load data
    health = pd.read_csv(health_path)
    supplements = pd.read_csv(supplement_path)
    experiments = pd.read_csv(experiments_path)
    profiles = pd.read_csv(profiles_path)

    # Clean sleep_hours: strip 'h'/'H' suffix and convert to float
    health['sleep_hours'] = health['sleep_hours'].str.replace('[hH]', '', regex=True).astype(float)

    # Parse dates
    health['date'] = pd.to_datetime(health['date']).dt.date
    supplements['date'] = pd.to_datetime(supplements['date']).dt.date

    # Convert dosage to grams
    supplements['dosage_grams'] = supplements.apply(
        lambda r: r['dosage'] / 1000 if r['dosage_unit'] == 'mg' else r['dosage'], axis=1
    )

    # Merge supplements with experiments to get experiment_name
    supplements = supplements.merge(
        experiments[['experiment_id', 'name']].rename(columns={'name': 'experiment_name'}),
        on='experiment_id', how='left'
    )

    # Merge health with supplements (left join - keep all health rows)
    merged = health.merge(
        supplements[['user_id', 'date', 'supplement_name', 'dosage_grams', 'is_placebo', 'experiment_name']],
        on=['user_id', 'date'], how='left'
    )

    # Fill missing supplement_name with 'No intake'
    merged['supplement_name'] = merged['supplement_name'].fillna('No intake')

    # Create age group bins
    def age_group(age):
        if pd.isna(age):
            return 'Unknown'
        age = int(age)
        if age < 18:
            return 'Under 18'
        elif age <= 25:
            return '18-25'
        elif age <= 35:
            return '26-35'
        elif age <= 45:
            return '36-45'
        elif age <= 55:
            return '46-55'
        elif age <= 65:
            return '56-65'
        else:
            return 'Over 65'

    profiles['user_age_group'] = profiles['age'].apply(age_group)

    # Merge with profiles
    merged = merged.merge(
        profiles[['user_id', 'email', 'user_age_group']],
        on='user_id', how='left'
    )

    # Select and order final columns
    result = merged[['user_id', 'date', 'email', 'user_age_group', 'experiment_name',
                      'supplement_name', 'dosage_grams', 'is_placebo',
                      'average_heart_rate', 'average_glucose', 'sleep_hours', 'activity_level']]

    return result

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = merge_all_data('user_health_data.csv', 'supplement_usage.csv', 'experiments.csv', 'user_profiles.csv')
df